# Session 3 - Choosing a model without fooling yourself

**Student notebook.** Cells marked **EXERCISE** are for you to fill in. Everything else can be run as it stands.

## Choosing a model without fooling yourself

You now have a trustworthy way to score one model. Today you have to choose
between many, and tune each of them — which creates a new way to be wrong, one
that no amount of careful cross-validation inside a single model can fix.

**This session (2 h)**

1. A comparison table with error bars, and a paired comparison that respects folds
2. Hyperparameter search, and how much of a leaderboard is noise
3. Nested cross-validation: scoring the *procedure*, not the winner
4. Learning curves: should you measure more molecules, or better descriptors?

In [ ]:
import sys, warnings
sys.path.insert(0, "..")          # so that `import qsar_helpers` works
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qsar_helpers import (load_fish, load_aquatic, xy, fold_error,
                          molecule_groups, score_table,
                          DESCRIPTOR_NOTES, TARGET_LABEL)

RNG = 0                            # one seed for the whole notebook
plt.rcParams.update({"figure.figsize": (6, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
print("pandas", pd.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (KFold, RepeatedKFold, GridSearchCV,
                                     cross_val_score, cross_validate, learning_curve)
from sklearn.base import clone
from sklearn.metrics import r2_score

fish = load_fish()
X, y = xy(fish)
cv_rep = RepeatedKFold(n_splits=5, n_repeats=5, random_state=RNG)   # 25 folds

## 1. The comparison table

### EXERCISE 3.1

Cross-validate six models with `cv_rep` (5-fold, 5 repeats = 25 folds):
dummy, linear regression, ridge (alpha=1), kNN (k=5), a random forest
(500 trees) and `HistGradientBoostingRegressor`. Scale where it matters.

Produce a table of mean R<sup>2</sup> ± sd and mean RMSE, and a horizontal bar
chart with error bars. Then answer: which models are distinguishable from each
other, given the spreads?

*Runtime about two minutes.*

In [ ]:
models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "LinearRegression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (a=1)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    "RandomForest": RandomForestRegressor(n_estimators=500, random_state=RNG),
    "HistGB": HistGradientBoostingRegressor(random_state=RNG),
}

# TODO: cross-validate each model with cv_rep, keep the per-fold R2 arrays in a
#       dict called `folds`, build the table, and plot mean +- sd
...

Dummy at -0.01, the two linear models tied at 0.56, and the three non-linear
models bunched between 0.60 and 0.61 — with fold-to-fold spreads of 0.05 to
0.06, i.e. larger than the gaps between them.

Two readings of that table are both correct and worth stating explicitly. The
step from "no model" to "linear model" is large and unambiguous. The step from
"linear" to "random forest" is about 0.05 in R<sup>2</sup>, which is the same
size as the error bars — so the bar chart alone cannot establish it.

### EXERCISE 3.2 — compare models on the folds they share

The fold-to-fold sd includes variation caused by the *folds* (some quarters of
this dataset are simply harder), and that variation is common to all models.
Because every model saw the same 25 folds, you can subtract it out: compute the
**per-fold difference** between two models instead of comparing their means.

1. Compute `folds["RandomForest"] - folds["Ridge (a=1)"]` and report its mean
   and sd. In what fraction of the 25 folds does the forest win?
2. Plot the paired fold scores (ridge on x, forest on y, with the diagonal) so
   the pairing is visible.
3. A paired t-test on these 25 differences would give a very small p-value.
   Explain why that p-value should not be reported.

In [ ]:
# TODO: paired per-fold differences, win fraction, paired scatter plot
...

The forest is ahead by 0.049 ± 0.041 per fold and wins 84% of folds: a real but
modest improvement, consistently in the same direction. That sentence is
defensible. "The random forest substantially outperformed linear models" is not.

## 2. Hyperparameter search

### EXERCISE 3.3

Use `GridSearchCV` to tune k for kNN over `[1, 2, 3, 5, 8, 12, 20, 30, 50]`.
Plot mean CV R<sup>2</sup> against k with its error bars, and report the best k.

In [ ]:
# TODO: GridSearchCV over n_neighbors; plot the validation curve with error bars
...

k=1 is badly overfit (R<sup>2</sup> ≈ 0.41: a single nearest neighbour is mostly
measurement noise), the curve peaks broadly around k=8-12, and large k drifts
back toward the dummy. The peak is flat — which is the point of the next
exercise.

### EXERCISE 3.4 — how much of a leaderboard is noise?

Build a list of model configurations: kNN for each k above, ridge for
alpha in `[0.01, 0.1, 1, 10, 100, 1000]`, random forests crossing
`max_depth` in `[2, 3, 5, 8, None]` with `max_features` in `[0.3, 0.6, 1.0]`,
and `HistGradientBoostingRegressor` crossing `learning_rate` in
`[0.03, 0.1, 0.3]` with `max_leaf_nodes` in `[15, 31, 63]`. That is 39
configurations.

Score every one with the **same** 5-fold split, sort them, and print the top
eight. Then compare the gap between 1st and 8th place with the ±0.045 sd of a
single split you measured in session 2.

*Runtime two to three minutes.*

In [ ]:
# TODO: build the 39-configuration dict, score all with one shared KFold,
#       print the top 8, compare the 1st-8th gap to the single-split sd
...

The top eight configurations span 0.023 in R<sup>2</sup> — half the sd of a single
train/test split, and they come from three different model families. The
identity of the "winner" is therefore mostly a property of this particular fold
assignment, not of the models. Re-run it with a different `random_state` in
`cv_shared` and a different configuration will be on top.

Two practical consequences. Reporting *only* the winner's score overstates what
you know. And the effort of squeezing out the last 0.02 is better spent on
better descriptors or more molecules — which is exactly what the learning curve
at the end of this session will tell you.

## 3. Nested cross-validation

The score above is contaminated in a way session 2 did not cover. We used the
same folds to *choose* the configuration and to *report* its performance. The
maximum of 39 noisy numbers is biased upward — the winner is partly whichever
configuration best fits that fold assignment's quirks.

Nested CV measures the whole procedure "try all 39, keep the best" instead:

- an **outer** split sets aside a test fold;
- within the remaining data, an **inner** CV picks the configuration;
- the winner is refit on the outer training part and scored once on the outer
  test fold;
- repeat for every outer fold.

### EXERCISE 3.5

Implement nested CV over the 39 configurations (5 outer folds, 5 inner folds)
and compare it with the flat "best score on the leaderboard" number.

*This fits 39 x 5 x 5 models and takes several minutes — start it, then read on.*

In [ ]:
# TODO: nested CV. For each outer fold: inner-CV all 39 configs on the outer
#       training part, clone the winner, refit, score on the outer test fold.
...

At n=908 the measured optimism is about **-0.01** — not detectable, and in fact
the nested estimate (0.645 ± 0.048) comes out slightly *above* the flat winner
(0.631). With a fold-to-fold sd of 0.048 that difference is noise; the honest
reading is that selection bias here is below what this experiment can resolve.

Worth noting what the outer folds chose: four of the five picked
`rf(max_depth=None, max_features=0.3)` and the fifth picked the same forest with
`max_features=0.6`. So the *family* is selected stably even though the
leaderboard positions within it are not — a useful distinction when someone
claims their architecture search found something.

### EXERCISE 3.6 — where selection bias does bite

Selection bias grows with the number of configurations tried and shrinks with
sample size. Repeat 3.5 on random subsamples of **120 molecules**, three times
with different subsamples, and report the mean optimism.

*Reduce the forests to `n_estimators=100` to keep this under a few minutes.*

In [ ]:
# TODO: three subsamples of 120 molecules; for each, flat best CV vs nested CV
...

At 120 molecules the same procedure is optimistic by **+0.062 on average**
(+0.126, +0.030, +0.029 across the three subsamples). Same code, same 39
configurations — only the sample size changed. Note also how much the three
subsamples differ: at this n, the optimism is itself a noisy quantity, and a
single estimate of it would not be worth much either.

The general rule: the inflation scales with the number of things you tried
divided by the amount of data you tried them on. Reported performance from a
small cohort after a large architecture or hyperparameter search should be read
as an upper bound, unless the paper says it nested the selection.

## 4. Learning curves: what to spend money on

A practical question with a real budget behind it. The model is wrong by a
factor of about eight in concentration. To improve it, should the lab measure
more molecules, or invest in better descriptors?

### EXERCISE 3.7

Use `learning_curve` for the random forest and for ridge regression, with
training sizes from 10% to 100%. Plot training and cross-validated R<sup>2</sup>
against training-set size for both. Then answer:

1. Which model is limited by variance (overfitting), and which by bias?
2. Is the forest's cross-validated score still rising at n=726?
3. What would you advise the lab to do?

In [ ]:
# TODO: learning curves for RF and Ridge; plot train vs CV score against n
...

The two curves say opposite things.

The **forest** sits at 0.94 on its training data and 0.60 in cross-validation.
That 0.34 gap is variance: the model is fitting molecule-specific noise it
cannot generalise. Its cross-validated score is still climbing at the full
dataset (0.52 at n=72 to 0.60 at n=726, with no plateau), so more molecules
would still help this model.

**Ridge** scores 0.58 on training data and 0.56 in cross-validation. There is
essentially no gap, and the curve flattens by about n=400. More molecules will
not help it at all: it is bias-limited, and the ceiling is set by what a linear
function of these six descriptors can express.

So the honest advice depends on which model you intend to deploy — and for the
flexible one, *both* more molecules and better descriptors would pay. The
descriptor question is the one session 4 opens up: six numbers cannot distinguish
molecules that differ in ways that matter for toxicity, which is also why 217 of
our rows have an identical twin.

## Session 3 checklist

1. Report every candidate's score, not only the winner's.
2. Compare models on shared folds, as paired differences; report effect size and
   win fraction, not a p-value from reused folds.
3. If you selected among configurations, either nest the selection or label the
   number an upper bound.
4. State the number of configurations you tried and the sample size, so a reader
   can judge the selection bias themselves.
5. Use a learning curve before asking for more data; it may not be the
   constraint.